# MRNet — 1. Download e organização dos dados

Este notebook:
1. baixa os 3.750 arquivos `.npy` do Redivis (7,7 GB) para o Colab;
2. monta os rótulos (corrigindo um problema dos CSVs no Redivis);
3. cria a **tabela única por exame** (`exames.csv`);
4. faz o **split 80/20 estratificado** do treino (`split.csv`);
5. mostra um exame de exemplo.

**Antes de rodar:** é preciso ter uma conta no Redivis com acesso aprovado ao dataset MRNet.
Na primeira chamada, a biblioteca `redivis` mostra um link para você entrar na sua conta (não use token de API no Colab: o Redivis desaconselha).

In [ ]:
!pip install -q redivis

In [ ]:
import os

# Pasta onde os dados vão ficar. O disco local do Colab é rápido;
# no fim tem uma célula opcional para guardar uma cópia no Drive.
ROOT = "/content/mrnet"
os.makedirs(ROOT, exist_ok=True)

## 1. Download dos arquivos

As tabelas `train` e `valid` do Redivis são índices de arquivos (`file_id`, `file_name`, `size`).
O código baixa cada arquivo para `ROOT/<split>/<plano>/<id>.npy`, em paralelo.
Se a conexão cair, é só rodar de novo: arquivos já baixados (com o tamanho certo) são pulados.

In [ ]:
import redivis
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

ds = redivis.organization("aimi").dataset("mrnet_knee_mri_s")

def baixar_um(tabela, nome, destino, tamanho):
    if os.path.exists(destino) and os.path.getsize(destino) == tamanho:
        return "pulado"
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    tabela.file(nome).download(destino, overwrite=True, progress=False)
    return "baixado"

def baixar_split(split, workers=16):
    tabela = ds.table(split)
    idx = tabela.to_pandas_dataframe()   # file_id, file_name, size, ...
    tarefas = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for _, r in idx.iterrows():
            destino = os.path.join(ROOT, split, r["file_name"])  # ex.: train/sagittal/0060.npy
            tarefas.append(ex.submit(baixar_um, tabela, r["file_name"], destino, int(r["size"])))
        erros = 0
        for f in tqdm(as_completed(tarefas), total=len(tarefas), desc=split):
            try:
                f.result()
            except Exception as e:
                erros += 1
                if erros <= 3:   # mostra só os primeiros, para não lotar a tela
                    print("erro:", e)
    print(f"{split}: {len(idx)} arquivos no índice, {erros} erros")
    return idx

idx_train = baixar_split("train")   # 3.390 arquivos (1.130 exames x 3 planos)
idx_valid = baixar_split("valid")   # 360 arquivos (120 exames x 3 planos)

In [ ]:
# Conferência: cada split precisa ter os 3 planos com o mesmo número de exames
for split, esperado in [("train", 1130), ("valid", 120)]:
    for plano in ["axial", "coronal", "sagittal"]:
        pasta = os.path.join(ROOT, split, plano)
        n = len([f for f in os.listdir(pasta) if f.endswith(".npy")])
        status = "ok" if n == esperado else f"ATENÇÃO: esperado {esperado}"
        print(f"{split}/{plano}: {n} arquivos  {status}")

## 2. Rótulos

Os CSVs originais do MRNet **não têm cabeçalho**. Quando foram importados no Redivis,
a primeira linha virou nome de coluna (por isso as tabelas têm 1.129 e 119 linhas, e não 1.130 e 120).
Ex.: em `train_abnormal` as colunas se chamam `_0000` e `_1`, que é o exame 0000 com rótulo 1.
A função abaixo recupera essa linha perdida.

In [ ]:
def carregar_rotulo(split, tarefa):
    df = ds.table(f"{split}_{tarefa}").to_pandas_dataframe()
    col_id, col_rot = df.columns[:2]
    df = df.rename(columns={col_id: "id", col_rot: tarefa})[["id", tarefa]]
    # recupera a 1ª linha que virou cabeçalho (ex.: "_0000", "_1")
    if str(col_id).startswith("_"):
        primeira = pd.DataFrame({"id": [str(col_id).lstrip("_")],
                                 tarefa: [int(str(col_rot).lstrip("_"))]})
        df = pd.concat([primeira, df], ignore_index=True)
    df["id"] = df["id"].astype(int)
    df[tarefa] = df[tarefa].astype(int)
    return df

rotulos = {}
for split, esperado in [("train", 1130), ("valid", 120)]:
    df = None
    for tarefa in ["abnormal", "acl", "meniscus"]:
        r = carregar_rotulo(split, tarefa)
        assert len(r) == esperado, f"{split}_{tarefa}: {len(r)} linhas, esperado {esperado}"
        df = r if df is None else df.merge(r, on="id", how="inner")
    assert len(df) == esperado
    df["split"] = split
    rotulos[split] = df

rotulos["train"].head()

## 3. Tabela única por exame

Uma linha por exame, com os 3 rótulos e o número de fatias de cada plano.
É a base para o Dataset do PyTorch e para a análise exploratória do relatório.

In [ ]:
import numpy as np

linhas = []
for split in ["train", "valid"]:
    for _, r in tqdm(rotulos[split].iterrows(), total=len(rotulos[split]), desc=split):
        lin = r.to_dict()
        for plano in ["axial", "coronal", "sagittal"]:
            caminho = os.path.join(ROOT, split, plano, f"{int(r['id']):04d}.npy")
            arr = np.load(caminho, mmap_mode="r")
            lin[f"fatias_{plano}"] = arr.shape[0]
            lin["altura"], lin["largura"] = arr.shape[1], arr.shape[2]
        linhas.append(lin)

exames = pd.DataFrame(linhas)
exames.to_csv(os.path.join(ROOT, "exames.csv"), index=False)
exames.head()

In [ ]:
# Conferências rápidas
print("Formato das imagens:", exames[["altura", "largura"]].drop_duplicates().values.tolist())
print()
print("Prevalência dos rótulos (%):")
print((exames.groupby("split")[["abnormal", "acl", "meniscus"]].mean() * 100).round(1))
print()
print("Nº de fatias por plano:")
print(exames[["fatias_axial", "fatias_coronal", "fatias_sagittal"]].describe().round(1))

## 4. Split treino / ajuste (80/20)

Os 20% de ajuste saem só do `train`. A `valid` oficial fica guardada para a avaliação final.
A estratificação usa a combinação dos 3 rótulos, para as proporções ficarem parecidas nos dois lados.
O arquivo `split.csv` deve ser usado por todo o grupo, para todo mundo treinar com a mesma divisão.

In [ ]:
from sklearn.model_selection import train_test_split

tr = exames[exames["split"] == "train"].copy()
combo = tr["abnormal"].astype(str) + tr["acl"].astype(str) + tr["meniscus"].astype(str)
print("Combinações de rótulos no treino:")
print(combo.value_counts())

ids_treino, ids_ajuste = train_test_split(
    tr["id"], test_size=0.20, random_state=42, stratify=combo
)

exames["subset"] = "valid_oficial"
exames.loc[exames["id"].isin(ids_treino) & (exames["split"] == "train"), "subset"] = "treino"
exames.loc[exames["id"].isin(ids_ajuste) & (exames["split"] == "train"), "subset"] = "ajuste"

exames[["id", "split", "subset"]].to_csv(os.path.join(ROOT, "split.csv"), index=False)
exames.to_csv(os.path.join(ROOT, "exames.csv"), index=False)

print()
print(exames["subset"].value_counts())
print()
print((exames.groupby("subset")[["abnormal", "acl", "meniscus"]].mean() * 100).round(1))

## 5. Olhando um exame

Fatia do meio de cada plano do primeiro exame de treino.

In [ ]:
import matplotlib.pyplot as plt

ex = exames.iloc[0]
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
for ax, plano in zip(axs, ["axial", "coronal", "sagittal"]):
    vol = np.load(os.path.join(ROOT, ex["split"], plano, f"{int(ex['id']):04d}.npy"))
    ax.imshow(vol[vol.shape[0] // 2], cmap="gray")
    ax.set_title(f"{plano} ({vol.shape[0]} fatias)")
    ax.axis("off")
fig.suptitle(f"Exame {int(ex['id']):04d}  anormal={ex['abnormal']}  LCA={ex['acl']}  menisco={ex['meniscus']}")
plt.show()

print("Tipo e faixa de valores:", vol.dtype, vol.min(), vol.max())

## 6. Guardar uma cópia no Google Drive

O disco do Colab é apagado quando a sessão termina. Para não baixar tudo de novo,
salvamos um único `mrnet.tar` (7,3 GiB) no Drive. O notebook 02 extrai dele.

O `tar` não consegue gravar direto no Drive montado ("Operation not supported"),
então o arquivo é criado no disco do Colab e depois copiado.
O `flush_and_unmount()` espera o envio para o Drive terminar (pode levar de 10 a 30 min). Não interrompa.

**Os dados não vão para o GitHub:** o acordo de uso do MRNet não permite redistribuir os exames.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!tar -cf /content/mrnet.tar -C /content mrnet
!cp /content/mrnet.tar /content/drive/MyDrive/mrnet.tar
!ls -lh /content/drive/MyDrive/mrnet.tar

drive.flush_and_unmount()